<a href="https://colab.research.google.com/github/mcfvdesigns/Site-Turnover-Defects/blob/main/M4%7CU3_Assignment_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import os, hashlib, zipfile, urllib.request, yaml
from pathlib import Path

DATA_URL = "https://github.com/mcfvdesigns/Site-Turnover-Defects/releases/download/v4.0/Site.Turnover.Defects.v4i.yolov11.1.zip"
SHA256 = "443eced074cc61c29b5fd3e0b8a0ce3bb5d6de410312312acc5ce936a10a84bc"
ROOT = Path("/content/dataset")
ZIP_PATH = Path("/content/dataset.zip")

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

def find_data_yaml(root: Path) -> Path:
    hits = list(root.rglob("data.yaml"))
    if not hits:
        raise FileNotFoundError(f"No data.yaml under {root}")
    return hits[0]

already_ready = (ROOT / "data.yaml").exists() or any(ROOT.glob("*/data.yaml"))
if not already_ready:
    ROOT.mkdir(parents=True, exist_ok=True)
    if ZIP_PATH.exists():
        ZIP_PATH.unlink()
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
    digest = sha256_file(ZIP_PATH)
    if digest != SHA256:
        ZIP_PATH.unlink(missing_ok=True)
        raise AssertionError(f"Checksum mismatch: {digest}")
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(ROOT)

cfg_path = find_data_yaml(ROOT)
dataset_root = cfg_path.parent
cfg = yaml.safe_load(cfg_path.read_text())
cfg["path"] = str(dataset_root)
cfg["train"] = "train/images"
cfg["val"] = "valid/images"
if (dataset_root / "test" / "images").is_dir():
    cfg["test"] = "test/images"
else:
    cfg.pop("test", None)
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(cfg)

{'train': 'train/images', 'val': 'valid/images', 'test': 'test/images', 'nc': 5, 'names': ['cracked jamb', 'cracked tile', 'moisture stain', 'paint defect', 'uneven joint'], 'roboflow': {'workspace': 'skipper-_sky', 'project': 'site-turnover-defects', 'version': 4, 'license': 'CC BY 4.0', 'url': 'https://universe.roboflow.com/skipper-_sky/site-turnover-defects/dataset/4'}, 'path': '/content/dataset'}


📌 Problem FramingDuring property handovers, manual inspections for architectural defects are time-consuming, subjective, and prone to human oversight.Objective: Build an automated instance segmentation pipeline to detect and delineate bounded defect boundaries on property surfaces.Classes Identified:cracked tilemoisture stainpaint defectuneven jointReal-World Impact: Reduces inspection resolution times, standardizes contractor quality control, and creates an audit trail prior to final client handoffs.

📊 Dataset Reference: The dataset was curated, annotated, and exported via Roboflow Universe / Workspace.Dataset Version: v4 (36 total images)Format: YOLOv8 / YOLO26 PyTorch SegmentationSplit Ratio: ~78% Train (28 images) | ~8% Validation (3 images) | ~14% Test (5 images)Preprocessing:Auto-orient appliedResized to $640 \times 640$ pixels (Stretched)📈 Training Outputs & Performance MetricsArchitecture: YOLO26 / YOLOv8 Instance Segmentation (Nano)Training Setup: 30+ Epochs on GPU hardware via Google Colab / Roboflow Cloud.Metric OverviewMetricScoreNotesmAP@50 (Mask)[Insert e.g., 0.12 or target]Highly constrained by small dataset size ($N=36$)Precision[Insert e.g., 0.15]Affected by class imbalance and background noiseRecall[Insert e.g., 0.10]Requires expanded training distributionNote: Training curves (results.png) and confusion matrices (confusion_matrix.png) are automatically exported to the runs/segment/train/ folder.

🔮 Inference & Validation Evidence. Inference outputs are evaluated across both the validation split and new, unseen site photos:Validation Predictions: Located in runs/segment/train/val_batch0_pred.jpgNew Images: Tested using Google Colab via: